In [2]:
import boto3

from sagemaker.core.helper.session_helper import (
    get_execution_role
)

AWS_REGION = "ap-south-1"

MODEL_PACKAGE_GROUP_NAME = (
    "beverage-price-prediction-xgboost"
)

MODEL_NAME = (
    "beverage-xgboost-v1-serverless-model"
)

ENDPOINT_CONFIG_NAME = (
    "beverage-xgboost-v1-serverless-config"
)

ENDPOINT_NAME = (
    "beverage-price-prediction-serverless"
)

sm = boto3.client(
    "sagemaker",
    region_name=AWS_REGION
)

role = get_execution_role()

print("Region:", AWS_REGION)
print("Role:", role)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Region: ap-south-1
Role: arn:aws:iam::812224290846:role/service-role/AmazonSageMaker-ExecutionRole-20260902T184223


In [3]:
packages = sm.list_model_packages(
    ModelPackageGroupName=
        MODEL_PACKAGE_GROUP_NAME,

    SortBy="CreationTime",
    SortOrder="Descending"
)

for pkg in packages["ModelPackageSummaryList"]:
    print(
        "Version:",
        pkg["ModelPackageVersion"],
        "| Approval:",
        pkg["ModelApprovalStatus"],
        "| ARN:",
        pkg["ModelPackageArn"]
    )

Version: 1 | Approval: Approved | ARN: arn:aws:sagemaker:ap-south-1:812224290846:model-package/beverage-price-prediction-xgboost/1


In [4]:
approved_packages = [
    pkg
    for pkg in packages["ModelPackageSummaryList"]
    if (
        pkg["ModelApprovalStatus"]
        == "Approved"
    )
]

assert approved_packages, (
    "No approved model version found"
)

MODEL_PACKAGE_ARN = (
    approved_packages[0][
        "ModelPackageArn"
    ]
)

print("Using approved model:")
print(MODEL_PACKAGE_ARN)

Using approved model:
arn:aws:sagemaker:ap-south-1:812224290846:model-package/beverage-price-prediction-xgboost/1


In [5]:
try:

    response = sm.create_model(
        ModelName=MODEL_NAME,

        ExecutionRoleArn=role,

        Containers=[
            {
                "ModelPackageName":
                    MODEL_PACKAGE_ARN
            }
        ]
    )

    print("✅ SageMaker Model created")
    print(response["ModelArn"])


except sm.exceptions.ClientError as e:

    if "already exists" in str(e).lower():
        print(
            "Model already exists:",
            MODEL_NAME
        )
    else:
        raise

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│    1 try:                                                                                        │
│    2 │                                                                                           │
│ ❱  3 │   response = sm.create_model(                                                             │
│    4 │   │   ModelName=MODEL_NAME,                                                               │
│    5 │   │                                                                                       │
│    6 │   │   ExecutionRoleArn=role,                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:606 in _api_call                      │
│                                                                                                  │
│    603 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    604 │   │   │   │   )                                                                         │
│    605 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  606 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    607 │   │                                                                                     │
│    608 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    609                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/context.py:123 in wrapper                       │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:1094 in _make_api_call                │
│                                                                                                  │
│   1091 │   │   │   │   'error_code_override'                                                     │
│   1092 │   │   │   ) or error_info.get("Code")                                                   │
│   1093 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1094 │   │   │   raise error_class(parsed_response, operation_name)                            │
│   1095 │   │   else:                                                                             │
│   1096 │   │   │   return parsed_response                                                        │
│   1097                                                                                           │
╰────────────────────────────────────────────────────────────

In [6]:
try:

    response = sm.create_endpoint_config(
        EndpointConfigName=
            ENDPOINT_CONFIG_NAME,

        ProductionVariants=[
            {
                "VariantName":
                    "AllTraffic",

                "ModelName":
                    MODEL_NAME,

                "ServerlessConfig": {
                    "MemorySizeInMB": 2048,
                    "MaxConcurrency": 1
                }
            }
        ]
    )

    print(
        "✅ Serverless endpoint "
        "configuration created"
    )

    print(
        response[
            "EndpointConfigArn"
        ]
    )


except sm.exceptions.ClientError as e:

    if "already exists" in str(e).lower():
        print(
            "Endpoint configuration "
            "already exists"
        )
    else:
        raise

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│    1 try:                                                                                        │
│    2 │                                                                                           │
│ ❱  3 │   response = sm.create_endpoint_config(                                                   │
│    4 │   │   EndpointConfigName=                                                                 │
│    5 │   │   │   ENDPOINT_CONFIG_NAME,                                                           │
│    6                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:606 in _api_call                      │
│                                                                                                  │
│    603 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    604 │   │   │   │   )                                                                         │
│    605 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  606 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    607 │   │                                                                                     │
│    608 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    609                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/context.py:123 in wrapper                       │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:1094 in _make_api_call                │
│                                                                                                  │
│   1091 │   │   │   │   'error_code_override'                                                     │
│   1092 │   │   │   ) or error_info.get("Code")                                                   │
│   1093 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1094 │   │   │   raise error_class(parsed_response, operation_name)                            │
│   1095 │   │   else:                                                                             │
│   1096 │   │   │   return parsed_response                                                        │
│   1097                                                                                           │
╰────────────────────────────────────────────────────────────

In [7]:
try:

    response = sm.create_endpoint(
        EndpointName=
            ENDPOINT_NAME,

        EndpointConfigName=
            ENDPOINT_CONFIG_NAME
    )

    print("✅ Endpoint creation started")
    print(response["EndpointArn"])


except sm.exceptions.ClientError as e:

    if "already exists" in str(e).lower():
        print(
            "Endpoint already exists:",
            ENDPOINT_NAME
        )
    else:
        raise

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│    1 try:                                                                                        │
│    2 │                                                                                           │
│ ❱  3 │   response = sm.create_endpoint(                                                          │
│    4 │   │   EndpointName=                                                                       │
│    5 │   │   │   ENDPOINT_NAME,                                                                  │
│    6                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:606 in _api_call                      │
│                                                                                                  │
│    603 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    604 │   │   │   │   )                                                                         │
│    605 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  606 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    607 │   │                                                                                     │
│    608 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    609                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/context.py:123 in wrapper                       │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:1094 in _make_api_call                │
│                                                                                                  │
│   1091 │   │   │   │   'error_code_override'                                                     │
│   1092 │   │   │   ) or error_info.get("Code")                                                   │
│   1093 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1094 │   │   │   raise error_class(parsed_response, operation_name)                            │
│   1095 │   │   else:                                                                             │
│   1096 │   │   │   return parsed_response                                                        │
│   1097                                                                                           │
╰────────────────────────────────────────────────────────────

In [7]:
import time

while True:

    endpoint = sm.describe_endpoint(
        EndpointName=ENDPOINT_NAME
    )

    status = endpoint["EndpointStatus"]

    print("Endpoint status:", status)

    if status == "InService":
        print("✅ Endpoint is ready")
        break

    if status == "Failed":
        print(
            "Failure reason:",
            endpoint.get(
                "FailureReason",
                "Unknown"
            )
        )
        break

    time.sleep(30)

Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Failed
Failure reason: Received server error (0) from model with message "An error occurred while handling request as the model process exited.". See https://ap-south-1.console.aws.amazon.com/cloudwatch/home?region=ap-south-1#logEventViewer:group=/aws/sagemaker/Endpoints/beverage-price-prediction-serverless in account 812224290846 for more information.


In [8]:
import boto3

logs = boto3.client(
    "logs",
    region_name="ap-south-1"
)

LOG_GROUP = (
    "/aws/sagemaker/Endpoints/"
    "beverage-price-prediction-serverless"
)

streams = logs.describe_log_streams(
    logGroupName=LOG_GROUP,
    orderBy="LastEventTime",
    descending=True,
    limit=5
)

for stream in streams["logStreams"]:
    print(stream["logStreamName"])

AllTraffic/63b23b4ca35731f25537b3bf98419826-4dff5d14a3724e52bef4325123850a82


In [9]:
for stream in streams["logStreams"][:3]:

    print("\n" + "=" * 80)
    print("STREAM:", stream["logStreamName"])
    print("=" * 80)

    events = logs.get_log_events(
        logGroupName=LOG_GROUP,
        logStreamName=stream["logStreamName"],
        startFromHead=False,
        limit=100
    )

    for event in events["events"]:
        print(event["message"])


STREAM: AllTraffic/63b23b4ca35731f25537b3bf98419826-4dff5d14a3724e52bef4325123850a82
2026-09-04 11:03:24,387 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)
2026-09-04 11:03:24,388 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)
2026-09-04 11:03:24,388 INFO - sagemaker-containers - nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
http {
  include /etc/nginx/mime.types;
  default_type application/octet-stream;
  access_log /dev/stdout combined;
  upstream gunicorn {
    server unix:/tmp/gunicorn.sock;
  }
  server {
    listen 8080 deferred;
    client_max_body_size 0;
    keepalive_timeout 3;
    location ~ ^/(ping|invocations|execution-parameters) {
      proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
      proxy_set_header Host $http_host;
      proxy_redirect off;
      proxy_read_timeout 60s;
      pro

## Redeployment — Model Registry Version 2

In [11]:
MODEL_NAME_V2 = (
    "beverage-xgboost-v2-serverless-model"
)

ENDPOINT_CONFIG_NAME_V2 = (
    "beverage-xgboost-v2-serverless-config"
)

ENDPOINT_NAME_V2 = (
    "beverage-price-prediction-serverless-v2"
)

In [12]:
packages = sm.list_model_packages(
    ModelPackageGroupName=
        MODEL_PACKAGE_GROUP_NAME,
    SortBy="CreationTime",
    SortOrder="Descending"
)

approved = [
    pkg
    for pkg in packages["ModelPackageSummaryList"]
    if pkg["ModelApprovalStatus"] == "Approved"
]

assert approved, "No approved model found"

MODEL_PACKAGE_ARN_V2 = approved[0][
    "ModelPackageArn"
]

print(
    "Deploying version:",
    approved[0]["ModelPackageVersion"]
)

print(MODEL_PACKAGE_ARN_V2)

Deploying version: 2
arn:aws:sagemaker:ap-south-1:812224290846:model-package/beverage-price-prediction-xgboost/2


In [13]:
response = sm.create_model(
    ModelName=MODEL_NAME_V2,
    ExecutionRoleArn=role,
    Containers=[
        {
            "ModelPackageName":
                MODEL_PACKAGE_ARN_V2
        }
    ]
)

print("✅ V2 SageMaker Model created")
print(response["ModelArn"])

✅ V2 SageMaker Model created
arn:aws:sagemaker:ap-south-1:812224290846:model/beverage-xgboost-v2-serverless-model


In [14]:
response = sm.create_endpoint_config(
    EndpointConfigName=
        ENDPOINT_CONFIG_NAME_V2,

    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": MODEL_NAME_V2,

            "ServerlessConfig": {
                "MemorySizeInMB": 2048,
                "MaxConcurrency": 1
            }
        }
    ]
)

print("✅ V2 endpoint config created")
print(response["EndpointConfigArn"])

✅ V2 endpoint config created
arn:aws:sagemaker:ap-south-1:812224290846:endpoint-config/beverage-xgboost-v2-serverless-config


In [15]:
response = sm.create_endpoint(
    EndpointName=ENDPOINT_NAME_V2,
    EndpointConfigName=
        ENDPOINT_CONFIG_NAME_V2
)

print("✅ V2 endpoint creation started")
print(response["EndpointArn"])

✅ V2 endpoint creation started
arn:aws:sagemaker:ap-south-1:812224290846:endpoint/beverage-price-prediction-serverless-v2


In [16]:
import time

while True:

    endpoint = sm.describe_endpoint(
        EndpointName=ENDPOINT_NAME_V2
    )

    status = endpoint["EndpointStatus"]

    print("Endpoint status:", status)

    if status == "InService":
        print("✅ V2 endpoint is ready")
        break

    if status == "Failed":
        print(
            "Failure reason:",
            endpoint.get(
                "FailureReason",
                "Unknown"
            )
        )
        break

    time.sleep(30)

Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Failed
Failure reason: Ping failed due to insufficient memory.


## V2 Redeployment — Increase Serverless Memory
### Step 1: Remove failed endpoint

In [17]:
import boto3
import time
from botocore.exceptions import ClientError

AWS_REGION = "ap-south-1"

ENDPOINT_NAME_V2 = (
    "beverage-price-prediction-serverless-v2"
)

sm = boto3.client(
    "sagemaker",
    region_name=AWS_REGION
)


# Check current status first
try:
    endpoint = sm.describe_endpoint(
        EndpointName=ENDPOINT_NAME_V2
    )

    print(
        "Current endpoint status:",
        endpoint["EndpointStatus"]
    )

except ClientError as e:
    print("Endpoint not found:", e)

Current endpoint status: Failed


In [18]:
sm.delete_endpoint(
    EndpointName=ENDPOINT_NAME_V2
)

print(
    "✅ Delete request submitted for:",
    ENDPOINT_NAME_V2
)

✅ Delete request submitted for: beverage-price-prediction-serverless-v2


In [19]:
while True:
    try:
        endpoint = sm.describe_endpoint(
            EndpointName=ENDPOINT_NAME_V2
        )

        print(
            "Waiting for deletion...",
            endpoint["EndpointStatus"]
        )

        time.sleep(15)

    except ClientError as e:

        if (
            e.response["Error"]["Code"]
            == "ValidationException"
        ):
            print("✅ Failed endpoint deleted")
            break

        raise

✅ Failed endpoint deleted


### Step 2: Create 4096 MB Serverless Endpoint Configuration

In [20]:
from botocore.exceptions import ClientError

MODEL_NAME_V2 = (
    "beverage-xgboost-v2-serverless-model"
)

ENDPOINT_CONFIG_NAME_V2_4096 = (
    "beverage-xgboost-v2-serverless-config-4096"
)

try:
    response = sm.create_endpoint_config(
        EndpointConfigName=ENDPOINT_CONFIG_NAME_V2_4096,

        ProductionVariants=[
            {
                "VariantName": "AllTraffic",

                "ModelName": MODEL_NAME_V2,

                "ServerlessConfig": {
                    "MemorySizeInMB": 4096,
                    "MaxConcurrency": 1
                }
            }
        ]
    )

    print("✅ 4096 MB endpoint configuration created")
    print(response["EndpointConfigArn"])

except ClientError as e:

    if "already" in str(e).lower():
        print(
            "Endpoint configuration already exists:",
            ENDPOINT_CONFIG_NAME_V2_4096
        )
    else:
        raise

✅ 4096 MB endpoint configuration created
arn:aws:sagemaker:ap-south-1:812224290846:endpoint-config/beverage-xgboost-v2-serverless-config-4096


In [21]:
config = sm.describe_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME_V2_4096
)

variant = config["ProductionVariants"][0]

print("Model:", variant["ModelName"])
print(
    "Memory:",
    variant["ServerlessConfig"]["MemorySizeInMB"],
    "MB"
)
print(
    "Max concurrency:",
    variant["ServerlessConfig"]["MaxConcurrency"]
)

Model: beverage-xgboost-v2-serverless-model
Memory: 4096 MB
Max concurrency: 1


### Step 3: Recreate V2 Endpoint with 4096 MB Memory

In [22]:
ENDPOINT_NAME_V2 = (
    "beverage-price-prediction-serverless-v2"
)

response = sm.create_endpoint(
    EndpointName=ENDPOINT_NAME_V2,
    EndpointConfigName=ENDPOINT_CONFIG_NAME_V2_4096
)

print("✅ Endpoint creation started")
print("Endpoint:", ENDPOINT_NAME_V2)
print("ARN:", response["EndpointArn"])

✅ Endpoint creation started
Endpoint: beverage-price-prediction-serverless-v2
ARN: arn:aws:sagemaker:ap-south-1:812224290846:endpoint/beverage-price-prediction-serverless-v2


In [23]:
import time

while True:
    endpoint = sm.describe_endpoint(
        EndpointName=ENDPOINT_NAME_V2
    )

    status = endpoint["EndpointStatus"]

    print("Endpoint status:", status)

    if status == "InService":
        print("✅ 4096 MB endpoint is ready")
        break

    if status == "Failed":
        print(
            "Failure reason:",
            endpoint.get(
                "FailureReason",
                "Unknown"
            )
        )
        break

    time.sleep(30)

Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Failed
Failure reason: Received server error (0) from model with message "An error occurred while handling request as the model process exited.". See https://ap-south-1.console.aws.amazon.com/cloudwatch/home?region=ap-south-1#logEventViewer:group=/aws/sagemaker/Endpoints/beverage-price-prediction-serverless-v2 in account 812224290846 for more information.


In [24]:
endpoint = sm.describe_endpoint(
    EndpointName=ENDPOINT_NAME_V2
)

print("Status:", endpoint["EndpointStatus"])
print("Config:", endpoint["EndpointConfigName"])
print("Failure:", endpoint.get("FailureReason"))

config = sm.describe_endpoint_config(
    EndpointConfigName=endpoint["EndpointConfigName"]
)

variant = config["ProductionVariants"][0]

print(
    "Memory:",
    variant["ServerlessConfig"]["MemorySizeInMB"],
    "MB"
)

print(
    "Model:",
    variant["ModelName"]
)

Status: Failed
Config: beverage-xgboost-v2-serverless-config-4096
Failure: Received server error (0) from model with message "An error occurred while handling request as the model process exited.". See https://ap-south-1.console.aws.amazon.com/cloudwatch/home?region=ap-south-1#logEventViewer:group=/aws/sagemaker/Endpoints/beverage-price-prediction-serverless-v2 in account 812224290846 for more information.
Memory: 4096 MB
Model: beverage-xgboost-v2-serverless-model


In [25]:
import boto3

logs = boto3.client(
    "logs",
    region_name="ap-south-1"
)

LOG_GROUP = (
    "/aws/sagemaker/Endpoints/"
    "beverage-price-prediction-serverless-v2"
)

streams = logs.describe_log_streams(
    logGroupName=LOG_GROUP,
    orderBy="LastEventTime",
    descending=True,
    limit=5
)

for i, stream in enumerate(streams["logStreams"]):
    print(
        i,
        stream["logStreamName"],
        stream.get("lastEventTimestamp")
    )

0 AllTraffic/954329c41b7e10bbab8d30c341eeb95b-962891c692fd4265bf4ff4f08376b64f 1788539953650
1 AllTraffic/e71e6ae1604a198f36a487b2fcb8192d-1d2492b7dd3d4f4980c426a0d74567aa 1788538719431


In [26]:
LATEST_STREAM = (
    streams["logStreams"][0]["logStreamName"]
)

print("Latest stream:")
print(LATEST_STREAM)

Latest stream:
AllTraffic/954329c41b7e10bbab8d30c341eeb95b-962891c692fd4265bf4ff4f08376b64f


In [27]:
events = logs.get_log_events(
    logGroupName=LOG_GROUP,
    logStreamName=LATEST_STREAM,
    startFromHead=True,
    limit=300
)

for event in events["events"]:
    print(event["message"])

2026-09-04 16:39:12,873 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)
2026-09-04 16:39:12,874 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)
2026-09-04 16:39:12,875 INFO - sagemaker-containers - nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
http {
  include /etc/nginx/mime.types;
  default_type application/octet-stream;
  access_log /dev/stdout combined;
  upstream gunicorn {
    server unix:/tmp/gunicorn.sock;
  }
  server {
    listen 8080 deferred;
    client_max_body_size 0;
    keepalive_timeout 3;
    location ~ ^/(ping|invocations|execution-parameters) {
      proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
      proxy_set_header Host $http_host;
      proxy_redirect off;
      proxy_read_timeout 60s;
      proxy_pass http://gunicorn;
    }
    location / {
      return 404 "{}";
    }
  }
}
202

## Deployment — Approved Model Version 3

### Step 1: Clean up failed V2 endpoint

In [28]:
from botocore.exceptions import ClientError

FAILED_ENDPOINT = (
    "beverage-price-prediction-serverless-v2"
)

try:
    endpoint = sm.describe_endpoint(
        EndpointName=FAILED_ENDPOINT
    )

    print(
        "Current status:",
        endpoint["EndpointStatus"]
    )

except ClientError as e:

    if e.response["Error"]["Code"] == "ValidationException":
        print("Endpoint already deleted.")
    else:
        raise

Current status: Failed


In [29]:
try:
    sm.delete_endpoint(
        EndpointName=FAILED_ENDPOINT
    )

    print("✅ Delete request submitted")

except ClientError as e:

    if e.response["Error"]["Code"] == "ValidationException":
        print("Endpoint already deleted.")
    else:
        raise

✅ Delete request submitted


In [30]:
import time

while True:

    try:
        endpoint = sm.describe_endpoint(
            EndpointName=FAILED_ENDPOINT
        )

        print(
            "Waiting for deletion...",
            endpoint["EndpointStatus"]
        )

        time.sleep(15)

    except ClientError as e:

        if e.response["Error"]["Code"] == "ValidationException":
            print("✅ Failed V2 endpoint deleted")
            break

        raise

✅ Failed V2 endpoint deleted


### Step 2: Create SageMaker Hosting Model from Approved V3

In [31]:
packages = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
    SortBy="CreationTime",
    SortOrder="Descending"
)

v3_packages = [
    pkg
    for pkg in packages["ModelPackageSummaryList"]
    if (
        pkg["ModelPackageVersion"] == 3
        and pkg["ModelApprovalStatus"] == "Approved"
    )
]

assert len(v3_packages) == 1, (
    "Approved Model Version 3 not found"
)

MODEL_PACKAGE_ARN_V3 = (
    v3_packages[0]["ModelPackageArn"]
)

print("Deploying Registry Version:", 3)
print("Approval:", v3_packages[0]["ModelApprovalStatus"])
print("ARN:", MODEL_PACKAGE_ARN_V3)

Deploying Registry Version: 3
Approval: Approved
ARN: arn:aws:sagemaker:ap-south-1:812224290846:model-package/beverage-price-prediction-xgboost/3


In [32]:
MODEL_NAME_V3 = (
    "beverage-xgboost-v3-serverless-model"
)

In [33]:
from botocore.exceptions import ClientError

try:
    existing = sm.describe_model(
        ModelName=MODEL_NAME_V3
    )

    print(
        "Model already exists:",
        MODEL_NAME_V3
    )

except ClientError as e:

    if e.response["Error"]["Code"] == "ValidationException":

        response = sm.create_model(
            ModelName=MODEL_NAME_V3,
            ExecutionRoleArn=role,
            Containers=[
                {
                    "ModelPackageName":
                        MODEL_PACKAGE_ARN_V3
                }
            ]
        )

        print("✅ V3 SageMaker Model created")
        print(response["ModelArn"])

    else:
        raise

✅ V3 SageMaker Model created
arn:aws:sagemaker:ap-south-1:812224290846:model/beverage-xgboost-v3-serverless-model


In [34]:
model_v3 = sm.describe_model(
    ModelName=MODEL_NAME_V3
)

print("Model:", MODEL_NAME_V3)

print(
    "Registry package:",
    model_v3["Containers"][0]["ModelPackageName"]
)

Model: beverage-xgboost-v3-serverless-model
Registry package: arn:aws:sagemaker:ap-south-1:812224290846:model-package/beverage-price-prediction-xgboost/3


### Step 3: Create V3 Serverless Endpoint Configuration

In [35]:
ENDPOINT_CONFIG_NAME_V3 = (
    "beverage-xgboost-v3-serverless-config-4096"
)

try:
    config = sm.describe_endpoint_config(
        EndpointConfigName=ENDPOINT_CONFIG_NAME_V3
    )

    print(
        "Endpoint config already exists:",
        ENDPOINT_CONFIG_NAME_V3
    )

except ClientError as e:

    if e.response["Error"]["Code"] == "ValidationException":

        response = sm.create_endpoint_config(
            EndpointConfigName=ENDPOINT_CONFIG_NAME_V3,

            ProductionVariants=[
                {
                    "VariantName": "AllTraffic",

                    "ModelName": MODEL_NAME_V3,

                    "ServerlessConfig": {
                        "MemorySizeInMB": 4096,
                        "MaxConcurrency": 1
                    }
                }
            ]
        )

        print("✅ V3 endpoint config created")
        print(response["EndpointConfigArn"])

    else:
        raise

✅ V3 endpoint config created
arn:aws:sagemaker:ap-south-1:812224290846:endpoint-config/beverage-xgboost-v3-serverless-config-4096


In [36]:
config_v3 = sm.describe_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME_V3
)

variant = config_v3["ProductionVariants"][0]

print("Endpoint config:", ENDPOINT_CONFIG_NAME_V3)
print("Model:", variant["ModelName"])
print(
    "Memory:",
    variant["ServerlessConfig"]["MemorySizeInMB"],
    "MB"
)
print(
    "Max concurrency:",
    variant["ServerlessConfig"]["MaxConcurrency"]
)

Endpoint config: beverage-xgboost-v3-serverless-config-4096
Model: beverage-xgboost-v3-serverless-model
Memory: 4096 MB
Max concurrency: 1


### Step 4: Create V3 Serverless Endpoint

In [37]:
ENDPOINT_NAME_V3 = (
    "beverage-price-prediction-serverless-v3"
)

In [38]:
try:
    existing = sm.describe_endpoint(
        EndpointName=ENDPOINT_NAME_V3
    )

    print(
        "Endpoint already exists:",
        ENDPOINT_NAME_V3,
        "| Status:",
        existing["EndpointStatus"]
    )

except ClientError as e:

    if e.response["Error"]["Code"] == "ValidationException":

        response = sm.create_endpoint(
            EndpointName=ENDPOINT_NAME_V3,
            EndpointConfigName=ENDPOINT_CONFIG_NAME_V3
        )

        print("✅ V3 endpoint creation started")
        print(response["EndpointArn"])

    else:
        raise

✅ V3 endpoint creation started
arn:aws:sagemaker:ap-south-1:812224290846:endpoint/beverage-price-prediction-serverless-v3


In [39]:
import time

while True:
    endpoint = sm.describe_endpoint(
        EndpointName=ENDPOINT_NAME_V3
    )

    status = endpoint["EndpointStatus"]

    print("Endpoint status:", status)

    if status == "InService":
        print("✅ V3 endpoint is ready")
        break

    if status == "Failed":
        print(
            "Failure reason:",
            endpoint.get(
                "FailureReason",
                "Unknown"
            )
        )
        break

    time.sleep(30)

Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Failed
Failure reason: Received server error (0) from model with message "An error occurred while handling request as the model process exited.". See https://ap-south-1.console.aws.amazon.com/cloudwatch/home?region=ap-south-1#logEventViewer:group=/aws/sagemaker/Endpoints/beverage-price-prediction-serverless-v3 in account 812224290846 for more information.


In [40]:
import boto3

logs = boto3.client(
    "logs",
    region_name="ap-south-1"
)

LOG_GROUP_V3 = (
    "/aws/sagemaker/Endpoints/"
    "beverage-price-prediction-serverless-v3"
)

streams_v3 = logs.describe_log_streams(
    logGroupName=LOG_GROUP_V3,
    orderBy="LastEventTime",
    descending=True,
    limit=3
)

for i, stream in enumerate(streams_v3["logStreams"]):
    print(i, stream["logStreamName"])

0 AllTraffic/5bf53feb0a9affe347d84970304691b1-9491b057b512434c80502d0d9e9b7f5c


In [41]:
LATEST_STREAM_V3 = (
    streams_v3["logStreams"][0]["logStreamName"]
)

events = logs.get_log_events(
    logGroupName=LOG_GROUP_V3,
    logStreamName=LATEST_STREAM_V3,
    startFromHead=True,
    limit=500
)

messages = [
    event["message"]
    for event in events["events"]
]

# Print the last ~100 log lines where the real crash
# normally appears.
for message in messages[-100:]:
    print(message)

  include /etc/nginx/mime.types;
  default_type application/octet-stream;
  access_log /dev/stdout combined;
  upstream gunicorn {
    server unix:/tmp/gunicorn.sock;
  }
  server {
    listen 8080 deferred;
    client_max_body_size 0;
    keepalive_timeout 3;
    location ~ ^/(ping|invocations|execution-parameters) {
      proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
      proxy_set_header Host $http_host;
      proxy_redirect off;
      proxy_read_timeout 60s;
      proxy_pass http://gunicorn;
    }
    location / {
      return 404 "{}";
    }
  }
}
2026-09-04 17:08:06,584 INFO - sagemaker-containers - Installing module with the following command:
/usr/bin/python -m pip install . -r requirements.txt
2026/09/04 17:08:07 [alert] 18#18: setrlimit(RLIMIT_NOFILE, 4096) failed (1: Operation not permitted)
2026/09/04 17:08:07 [alert] 18#18: prctl(PR_SET_DUMPABLE) failed (1: Operation not permitted)
2026/09/04 17:08:07 [alert] 20#20: setrlimit(RLIMIT_NOFILE, 4096) failed (1:

## V3 Hosting Runtime Path Fix

CloudWatch confirmed that the V3 package and dependencies install
successfully, but the serving process cannot import the installed
inference module.

Fix:
- install runtime packages into a known model-code directory
- explicitly expose that directory through PYTHONPATH
- reuse approved Model Registry Version 3

In [42]:
MODEL_NAME_V3_FIXED = (
    "beverage-xgboost-v3-serverless-model-fixed"
)

RUNTIME_SITE_PACKAGES = (
    "/opt/ml/model/code/site-packages"
)

RUNTIME_PYTHONPATH = (
    "/opt/ml/model/code:"
    "/opt/ml/model/code/site-packages"
)

In [43]:
from botocore.exceptions import ClientError

try:
    existing = sm.describe_model(
        ModelName=MODEL_NAME_V3_FIXED
    )

    print(
        "Model already exists:",
        MODEL_NAME_V3_FIXED
    )

except ClientError as e:

    if e.response["Error"]["Code"] == "ValidationException":

        response = sm.create_model(
            ModelName=MODEL_NAME_V3_FIXED,

            ExecutionRoleArn=role,

            Containers=[
                {
                    "ModelPackageName":
                        MODEL_PACKAGE_ARN_V3,

                    "Environment": {
                        "SAGEMAKER_PROGRAM":
                            "inference.py",

                        "SAGEMAKER_SUBMIT_DIRECTORY":
                            "/opt/ml/model/code",

                        "PIP_BREAK_SYSTEM_PACKAGES":
                            "1",

                        "PIP_TARGET":
                            RUNTIME_SITE_PACKAGES,

                        "PYTHONPATH":
                            RUNTIME_PYTHONPATH,
                    }
                }
            ]
        )

        print(
            "✅ V3 fixed hosting model created"
        )
        print(response["ModelArn"])

    else:
        raise

✅ V3 fixed hosting model created
arn:aws:sagemaker:ap-south-1:812224290846:model/beverage-xgboost-v3-serverless-model-fixed


In [44]:
fixed_model = sm.describe_model(
    ModelName=MODEL_NAME_V3_FIXED
)

container = fixed_model["Containers"][0]

print("Model:", MODEL_NAME_V3_FIXED)
print(
    "Registry package:",
    container["ModelPackageName"]
)

print("\nRuntime environment:")

for key, value in (
    container.get("Environment", {})
    .items()
):
    print(f"{key}: {value}")

Model: beverage-xgboost-v3-serverless-model-fixed
Registry package: arn:aws:sagemaker:ap-south-1:812224290846:model-package/beverage-price-prediction-xgboost/3

Runtime environment:
PIP_BREAK_SYSTEM_PACKAGES: 1
PIP_TARGET: /opt/ml/model/code/site-packages
PYTHONPATH: /opt/ml/model/code:/opt/ml/model/code/site-packages
SAGEMAKER_PROGRAM: inference.py
SAGEMAKER_SUBMIT_DIRECTORY: /opt/ml/model/code


### Create Endpoint Configuration for Fixed V3 Hosting Model

In [45]:
ENDPOINT_CONFIG_NAME_V3_FIXED = (
    "beverage-xgboost-v3-fixed-serverless-config-4096"
)

try:
    existing_config = sm.describe_endpoint_config(
        EndpointConfigName=ENDPOINT_CONFIG_NAME_V3_FIXED
    )

    print(
        "Endpoint config already exists:",
        ENDPOINT_CONFIG_NAME_V3_FIXED
    )

except ClientError as e:

    if e.response["Error"]["Code"] == "ValidationException":

        response = sm.create_endpoint_config(
            EndpointConfigName=ENDPOINT_CONFIG_NAME_V3_FIXED,

            ProductionVariants=[
                {
                    "VariantName": "AllTraffic",

                    "ModelName": MODEL_NAME_V3_FIXED,

                    "ServerlessConfig": {
                        "MemorySizeInMB": 4096,
                        "MaxConcurrency": 1
                    }
                }
            ]
        )

        print("✅ Fixed V3 endpoint config created")
        print(response["EndpointConfigArn"])

    else:
        raise

✅ Fixed V3 endpoint config created
arn:aws:sagemaker:ap-south-1:812224290846:endpoint-config/beverage-xgboost-v3-fixed-serverless-config-4096


In [46]:
config = sm.describe_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME_V3_FIXED
)

variant = config["ProductionVariants"][0]

print("Config:", ENDPOINT_CONFIG_NAME_V3_FIXED)
print("Model:", variant["ModelName"])
print(
    "Memory:",
    variant["ServerlessConfig"]["MemorySizeInMB"],
    "MB"
)
print(
    "Max concurrency:",
    variant["ServerlessConfig"]["MaxConcurrency"]
)

Config: beverage-xgboost-v3-fixed-serverless-config-4096
Model: beverage-xgboost-v3-serverless-model-fixed
Memory: 4096 MB
Max concurrency: 1


### Remove Failed V3 Endpoint Before Redeployment

In [47]:
FAILED_ENDPOINT_V3 = (
    "beverage-price-prediction-serverless-v3"
)

try:
    endpoint = sm.describe_endpoint(
        EndpointName=FAILED_ENDPOINT_V3
    )

    print(
        "Current status:",
        endpoint["EndpointStatus"]
    )

except ClientError as e:

    if e.response["Error"]["Code"] == "ValidationException":
        print("Endpoint already deleted.")
    else:
        raise

Current status: Failed


In [48]:
try:
    sm.delete_endpoint(
        EndpointName=FAILED_ENDPOINT_V3
    )

    print("✅ Delete request submitted")

except ClientError as e:

    if e.response["Error"]["Code"] == "ValidationException":
        print("Endpoint already deleted.")
    else:
        raise

✅ Delete request submitted


In [49]:
import time

while True:
    try:
        endpoint = sm.describe_endpoint(
            EndpointName=FAILED_ENDPOINT_V3
        )

        print(
            "Waiting for deletion...",
            endpoint["EndpointStatus"]
        )

        time.sleep(15)

    except ClientError as e:

        if e.response["Error"]["Code"] == "ValidationException":
            print("✅ Failed V3 endpoint deleted")
            break

        raise

✅ Failed V3 endpoint deleted


### Recreate V3 Endpoint Using Fixed Runtime Configuration

In [50]:
ENDPOINT_NAME_V3 = (
    "beverage-price-prediction-serverless-v3"
)

response = sm.create_endpoint(
    EndpointName=ENDPOINT_NAME_V3,
    EndpointConfigName=ENDPOINT_CONFIG_NAME_V3_FIXED
)

print("✅ Fixed V3 endpoint creation started")
print(response["EndpointArn"])

✅ Fixed V3 endpoint creation started
arn:aws:sagemaker:ap-south-1:812224290846:endpoint/beverage-price-prediction-serverless-v3


In [51]:
import time

while True:
    endpoint = sm.describe_endpoint(
        EndpointName=ENDPOINT_NAME_V3
    )

    status = endpoint["EndpointStatus"]

    print("Endpoint status:", status)

    if status == "InService":
        print("✅ Fixed V3 endpoint is ready")
        break

    if status == "Failed":
        print(
            "Failure reason:",
            endpoint.get(
                "FailureReason",
                "Unknown"
            )
        )
        break

    time.sleep(30)

Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: InService
✅ Fixed V3 endpoint is ready


### Step 5: Invoke Live SageMaker Serverless Endpoint

In [52]:
import boto3
import json
import io
import pandas as pd

runtime = boto3.client(
    "sagemaker-runtime",
    region_name="ap-south-1"
)

print("Endpoint:", ENDPOINT_NAME_V3)

Endpoint: beverage-price-prediction-serverless-v3


In [53]:
s3 = boto3.client(
    "s3",
    region_name="ap-south-1"
)

bucket = "krushang-beverage-ml-2026"

processed_key = (
    "processed/"
    "cleaned_survey_results.csv"
)

data_bytes = s3.get_object(
    Bucket=bucket,
    Key=processed_key
)["Body"].read()

df_live_test = pd.read_csv(
    io.BytesIO(data_bytes)
)

print("Shape:", df_live_test.shape)
print(df_live_test.columns.tolist())

Shape: (29956, 18)
['respondent_id', 'age', 'gender', 'zone', 'occupation', 'income_levels', 'consume_frequency(weekly)', 'current_brand', 'preferable_consumption_size', 'awareness_of_other_brands', 'reasons_for_choosing_brands', 'flavor_preference', 'purchase_channel', 'packaging_preference', 'health_concerns', 'typical_consumption_situations', 'price_range', 'age_group']


In [54]:
sample_live = (
    df_live_test
    .drop(
        columns=["price_range"],
        errors="ignore"
    )
    .head(1)
    .copy()
)

sample_live

,respondent_id,age,gender,zone,occupation,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,reasons_for_choosing_brands,flavor_preference,purchase_channel,packaging_preference,health_concerns,typical_consumption_situations,age_group
0,R00001,30,M,Urban,Working Professional,<10L,3-4 times,Newcomer,Medium (500 ml),0 to 1,Price,Traditional,Online,Simple,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",26-35


In [56]:
records = json.loads(
    sample_live.to_json(
        orient="records"
    )
)

payload = {
    "instances": records
}

request_body = json.dumps(payload)

print(
    json.dumps(
        payload,
        indent=2
    )
)

{
  "instances": [
    {
      "respondent_id": "R00001",
      "age": 30,
      "gender": "M",
      "zone": "Urban",
      "occupation": "Working Professional",
      "income_levels": "<10L",
      "consume_frequency(weekly)": "3-4 times",
      "current_brand": "Newcomer",
      "preferable_consumption_size": "Medium (500 ml)",
      "awareness_of_other_brands": "0 to 1",
      "reasons_for_choosing_brands": "Price",
      "flavor_preference": "Traditional",
      "purchase_channel": "Online",
      "packaging_preference": "Simple",
      "health_concerns": "Medium (Moderately health-conscious)",
      "typical_consumption_situations": "Active (eg. Sports, gym)",
      "age_group": "26-35"
    }
  ]
}


In [57]:
response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME_V3,
    ContentType="application/json",
    Accept="application/json",
    Body=request_body
)

response_body = (
    response["Body"]
    .read()
    .decode("utf-8")
)

result = json.loads(response_body)

print(
    json.dumps(
        result,
        indent=2
    )
)

{
  "predictions": [
    {
      "predicted_class": 1,
      "predicted_price_range": "100-150",
      "confidence": 0.9726226925849915,
      "probabilities": {
        "50-100": 1.666276148171164e-05,
        "100-150": 0.9726226925849915,
        "150-200": 0.02736065536737442,
        "200-250": 3.2431091145923574e-09
      }
    }
  ]
}
